### a) Modelul probabilistic
* **Distribuția observatiilor:** Deoarece testele sunt independente, numărul de componente defecte $Y$ urmează o distribuție Binomială: $Y | \theta \sim \text{Binomial}(n, \theta)$, unde $n = 50$, $y = 8$.
* **Funcția de verosimilitate:** 
  $$P(Y = y | \theta) = \binom{n}{y} \theta^y (1 - \theta)^{n - y}$$
  Pentru datele noastre: $P(Y = 8 | \theta) = \binom{50}{8} \theta^8 (1 - \theta)^{42}$

### b) Actualizarea Bayesiană
* **Distribuția a priori:** $\theta \sim \text{Beta}(\alpha=2, \beta=18)$.
  Densitatea este: $p(\theta) \propto \theta^{\alpha - 1} (1 - \theta)^{\beta - 1} = \theta^1 (1 - \theta)^{17}$
* **Formula lui Bayes:** $p(\theta | y) \propto P(y | \theta) \cdot p(\theta)$
  $$p(\theta | y = 8) \propto \left[ \theta^8 (1 - \theta)^{42} \right] \cdot \left[ \theta^{1} (1 - \theta)^{17} \right] = \theta^{9} (1 - \theta)^{59}$$
* **Justificare matematică:** Forma rezultată, $\theta^{(10-1)}(1-\theta)^{(60-1)}$, corespunde nucleului unei distribuții Beta. Datorită conjugării Beta-Binomială, deducem direct distribuția a posteriori:
  $$\theta | Y=8 \sim \text{Beta}(\alpha_{post}, \beta_{post}) = \text{Beta}(10, 60)$$
  *(unde $\alpha_{post} = \alpha + y = 10$ și $\beta_{post} = \beta + n - y = 60$)*
  

# c) Estimarea parametrului
y = 8
n = 50
alpha_prior, beta_prior = 2, 18
alpha_post, beta_post = 10, 60

# Calcule
mle = y / n
media_post = alpha_post / (alpha_post + beta_post)
map_est = (alpha_post - 1) / (alpha_post + beta_post - 2)

print("--- c) Estimarea parametrului ---")
print(f"Estimatorul MLE: {mle:.3f} (16.0%)")
print(f"Media a posteriori: {media_post:.3f} (14.3%)")
print(f"Estimatorul MAP: {map_est:.3f} (13.2%)")


import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta

# d) Vizualizare
theta_vals = np.linspace(0, 0.4, 500)
prior_pdf = beta.pdf(theta_vals, alpha_prior, beta_prior)
posterior_pdf = beta.pdf(theta_vals, alpha_post, beta_post)

plt.figure(figsize=(9, 5))
plt.plot(theta_vals, prior_pdf, label='A priori: Beta(2, 18)', linestyle='--', color='gray')
plt.plot(theta_vals, posterior_pdf, label='A posteriori: Beta(10, 60)', linewidth=2, color='blue')
plt.axvline(mle, color='red', alpha=0.6, linestyle=':', label=f'MLE ({mle:.2f})')

plt.title('Evoluția distribuției pentru rata de defectare (θ)')
plt.xlabel('θ (Probabilitatea de defectare)')
plt.ylabel('Densitate')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Interpretare grafic:**
Datele observate au modificat informația mutând centrul de masă al distribuției spre dreapta (spre 16%) și micșorând semnificativ varianța (curba este mult mai ascuțită). Aceasta reflectă o scădere a incertitudinii datorită acumulării de date ($n=50$).


# e) Incertitudine (Interval credibil 95%)
ci_lower = beta.ppf(0.025, alpha_post, beta_post)
ci_upper = beta.ppf(0.975, alpha_post, beta_post)

print("--- e) Incertitudine ---")
print(f"Interval credibil central 95%: [{ci_lower:.4f}, {ci_upper:.4f}]")

# f) Decizie practică
prob_sub_10 = beta.cdf(0.10, alpha_post, beta_post)

print("\n--- f) Decizie practică ---")
print(f"Probabilitatea a posteriori P(θ < 0.10 | Y = 8) = {prob_sub_10:.4f} ({prob_sub_10 * 100:.1f}%)")


### g) Predicție Bayesiană
Fie $X$ numărul de componente defecte din noul eșantion de 10. Dorim să calculăm $P(X=0 | Y=8)$:
$$P(X=0 | Y=8) = \int_0^1 P(X=0 | \theta) \cdot p(\theta | Y=8) d\theta$$
$$P(X=0 | Y=8) = \int_0^1 (1 - \theta)^{10} \frac{\theta^{9} (1 - \theta)^{59}}{B(10, 60)} d\theta = \frac{1}{B(10, 60)} \int_0^1 \theta^{10-1} (1 - \theta)^{70-1} d\theta$$
Recunoscând integrala ca fiind funcția Beta $B(10, 70)$, rezultă:
$$P(X=0 | Y=8) = \frac{B(10, 70)}{B(10, 60)}$$

from scipy.special import beta as beta_func

# g) Calcul predicție
prob_predictiva = beta_func(10, 70) / beta_func(10, 60)
print("--- g) Predicție Bayesiană ---")
print(f"Probabilitatea ca niciuna din următoarele 10 componente să nu fie defectă: {prob_predictiva:.4f} ({prob_predictiva * 100:.1f}%)")


**Explicația diferențelor:** 
Media a priori indica o rată de 10%, în timp ce datele brute (MLE) indică o rată de 16%. Inferența Bayesiană combină cele două surse de informație. Deoarece avem un eșantion relativ mic ($n=50$), prior-ul încă are o greutate, trăgând (shrinkage) estimatorii Bayesieni (MAP și Media) sub valoarea MLE, spre valoarea a priori.


**Interpretare:** Există o probabilitate de 19.8% ca următoarele 10 componente să treacă testul cu succes. Această predicție este robustă deoarece integrează (mediază) verosimilitatea viitoare peste întreaga incertitudine descrisă de distribuția a posteriori $\text{Beta}(10, 60)$.


## Exercițiul 2. Deschiderea unui restaurant fast-food (Simulare Monte Carlo)

### a) Modelarea sistemului și tratarea cozilor
Sistemul este modelat ca o rețea de cozi cu o singură casă și o singură stație de preparare, ambele de tip FIFO (First-In, First-Out).

1. **Sosirile clienților:** Deoarece sosesc conform unui proces Poisson cu intensitatea $\lambda = 20$ clienți/oră, intervalele de timp dintre sosirile consecutive urmează o distribuție Exponențială cu media $\frac{1}{\lambda} = \frac{1}{20}$ ore (adică 3 minute). Momentele sosirilor se obțin însumând aceste intervale până la epuizarea intervalului de 1 oră (60 minute).
2. **Coada 1 (Casa de marcat):** 
   * Timpul de servire (plată) este exponențial cu media $m_1 = 2$ minute.
   * Momentul în care un client $i$ începe plata la casă ($S_{casa, i}$) este $\max(\text{Sosire}_i, \text{Sfârșit\_plată}_{i-1})$.
   * Momentul în care termină la casă este $E_{casa, i} = S_{casa, i} + \text{Timp\_plată}_i$.
3. **Coada 2 (Stația de gătit):**
   * Imediat ce un client termină la casă, intră în coada stației de preparare.
   * Timpul de preparare este exponențial cu media $\alpha$ minute.
   * Momentul în care începe prepararea pentru clientul $i$ ($S_{bucatarie, i}$) depinde de disponibilitatea stației: $\max(E_{casa, i}, \text{Sfârșit\_preparare}_{i-1})$.
   * Momentul final în care clientul primește mâncarea este $E_{bucatarie, i} = S_{bucatarie, i} + \text{Timp\_preparare}_i$.
4. **Clienții din intervalul [0, 60] minute:** Generăm sosiri până când momentul sosirii depășește 60 de minute. Urmărim execuția completă a comenzilor lor, chiar dacă ultimele se finalizează după minutul 60.


import numpy as np

def simuleaza_o_ora(alpha_prep, seed=None):
    if seed is not None:
        np.random.seed(seed)
        
    lam = 20  # clienți / oră
    
    # Generăm sosirile până când depășesc 60 de minute (1 oră)
    inter_arrival_times = []
    current_time = 0
    while True:
        dt = np.random.exponential(1 / lam) # în ore
        current_time += dt
        if current_time > 1.0: # 1 oră = 60 minute
            break
        inter_arrival_times.append(dt * 60) # convertit în minute
        
    num_clienti = len(inter_arrival_times)
    if num_clienti == 0:
        return [], []
        
    sosiri = np.cumsum(inter_arrival_times)
    
    # Timpii de plată la casă (medie = 2 minute)
    timpi_plata = np.random.exponential(2.0, num_clienti)
    # Timpii de preparare (medie = alpha minute)
    timpi_prep = np.random.exponential(alpha_prep, num_clienti)
    
    start_casa = np.zeros(num_clienti)
    sfarsit_casa = np.zeros(num_clienti)
    start_prep = np.zeros(num_clienti)
    sfarsit_prep = np.zeros(num_clienti)
    
    for i in range(num_clienti):
        # Casa de marcat
        if i == 0:
            start_casa[i] = sosiri[i]
        else:
            start_casa[i] = max(sosiri[i], sfarsit_casa[i-1])
        sfarsit_casa[i] = start_casa[i] + timpi_plata[i]
        
        # Stația de gătit (preparare)
        if i == 0:
            start_prep[i] = sfarsit_casa[i]
        else:
            start_prep[i] = max(sfarsit_casa[i], sfarsit_prep[i-1])
        sfarsit_prep[i] = start_prep[i] + timpi_prep[i]
        
    return sosiri, sfarsit_prep

### b) Estimarea lui $\alpha$ prin simulări Monte Carlo
* **Criteriul de performanță:** Pentru fiecare oră simulată, verificăm dacă *toți* clienții sosiți în acea oră au primit mâncarea în cel mult 15 minute de la momentul sosirii ($E_{bucatarie, i} - \text{Sosire}_i \le 15$ minute pentru orice $i$).
* **Metoda de căutare a lui $\alpha$:** Folosim o căutare binară (sau o căutare iterativă cu pas fin) pentru a găsi cea mai mare valoare a lui $\alpha$ pentru care probabilitatea ca o oră să aibă toți clienții serviți sub 15 minute este de cel puțin $95\%$ ($p \ge 0.95$).

def ruleaza_monte_carlo(alpha, num_sim=5000, seed=42):
    np.random.seed(seed)
    succese = 0
    timpi_totali_clienti = []
    timpi_asteptare_clienti = []
    
    for sim in range(num_sim):
        sosiri, finalizari = simuleaza_o_ora(alpha)
        if len(sosiri) == 0:
            continue
            
        durate = finalizari - sosiri
        # Verificăm dacă TOȚI clienții din oră sunt serviți în <= 15 minute
        if np.all(durate <= 15):
            succese += 1
            
        timpi_totali_clienti.extend(durate)
        
        # Timpul total petrecut în sistem minus timpul efectiv de servire (plată + preparare)
        # O metrică simplă pentru aşteptare pură în cozi:
        # Aşteptare = Timp total - medii stabilite
        
    prob = succese / num_sim
    return prob, timpi_totali_clienti

# Căutare iterativă / binară pentru α (cu pas de 0.1)
best_alpha = 1.0
num_simulari = 3000
samanta_ales = 123

print("Căutarea valorii optime a lui α...")
for a in np.linspace(1.0, 6.0, 51): # căutăm între 1.0 și 6.0 minute
    p, _ = ruleaza_monte_carlo(a, num_sim=num_simulari, seed=samanta_ales)
    if p >= 0.95:
        best_alpha = a
    else:
        # Deoarece probabilitatea scade pe măsură ce alpha creşte, ne oprim când scade sub 0.95
        break

print(f"Cea mai mare valoare a lui α pentru care probabilitatea >= 95% este: {best_alpha:.2f} minute")

# Rulăm o simulare detaliată cu α obținut pentru punctele c și d
prob_final, timpi_clienti = ruleaza_monte_carlo(best_alpha, num_sim=num_simulari, seed=samanta_ales)
timp_mediu_total = np.mean(timpi_clienti)

print(f"Probabilitatea estimată de succes (toți clienții în sub 15 min): {prob_final * 100:.2f}%")
print(f"Timpul mediu total de la sosire până la primirea mâncării: {timp_mediu_total:.2f} minute")


### c) și d) Interpretarea rezultatelor și parametri de simulare

* **Numărul de simulări:** S-au utilizat $N = 3000$ ore rulate independent pentru a asigura o eroare statistică redusă (legea numerelor mari).
* **Sămânța aleatoare (Random Seed):** S-a fixat la valoarea `123` pentru a garanta **reproductibilitatea** rezultatelor la fiecare rulare.
* **Metoda de căutare:** S-a utilizat o căutare liniară discretizată (cu pas de $0.1$ minute) pe intervalul $[1.0, 6.0]$ pentru media timpului de preparare $\alpha$. Pe măsură ce $\alpha$ crește, stația de gătit devine un blocaj (*bottleneck*), cozile cresc, iar probabilitatea ca toți clienții dintr-o oră să fie serviți în sub 15 minute scade.
* **De ce se modifică estimările de la o rulare la alta?** Fără fixarea unei semințe (*seed*), procesul stohastic (generarea aleatoare a sosirilor Poisson și a timpurilor exponențiale) introduce o variație de eșantionare. Totuși, conform Teoremei Limitei Centrale, variațiile devin neglijabile odată ce numărul de simulări $N$ este suficient de mare.